In [14]:
import csv
import sys
import pandas as pd
sys.path.append('/home/unix/vkrhk/EmmaEmb/EmmaEmb/analysis')
from constants import IMG_OUTPUT_PATH, EMB_SPACES

PRECOMPUTED_KNN_PATH = f'{IMG_OUTPUT_PATH}/knn-binding-sites'
def read_kNN(k, emb_space_name, metric, do_average=True):
    results = {}
    with open(f'{PRECOMPUTED_KNN_PATH}/{metric}/{emb_space_name},k={k}.csv', 'r') as f:
        knn_results = csv.reader(f, delimiter=',')
        next(knn_results)
        for row in knn_results:
            feature, acc = row[0], float(row[1])
            class_name, number_of_points = feature.split(' ')[0], int(feature.split(' ')[-1][:-1])
            results[class_name] = (acc, number_of_points)
        
    if do_average:
        results = sum(float(a) * n for a, n in results.values()) / sum(n for _, n in results.values())
        return results
    else:
        return results

print(read_kNN(3, 'ESM1', 'euclidean'))
print(read_kNN(5, 'ESM1', 'euclidean'))
print(read_kNN(10, 'ESM1', 'euclidean'))
print(read_kNN(50, 'ESM1', 'euclidean'))
print(read_kNN(100, 'ESM1', 'euclidean'))
print(read_kNN(200, 'ESM1', 'euclidean'))

0.880748069874236
0.8751334885761968
0.8669223796360654
0.85366890989401
0.8513300491032961
0.8505583137678212


In [22]:
Ks = [3, 5, 10, 50, 100, 200]

collected_results = []
for metric in ['euclidean', 'cosine', 'cityblock']:
    for k in Ks:
        for emb_space in EMB_SPACES:
            collected_results.append((k, emb_space[0], read_kNN(k, emb_space[0], metric), metric))

df = pd.DataFrame(collected_results, columns=['k', 'Embedding Space', 'kNN score', 'metric'])


In [23]:
import plotly.express as px
def plot_kNN_alignment_scores(fig):
    fig.update_layout(
            yaxis_title="Mean KNN feature<br>alignment score",
            template="plotly_white",
            font={"family": "Arial", "color": "black", "size": 12},
            legend_title_text="Embedding models",
            width=1200,
            height=500,
    )

    fig.update_traces(marker=dict(size=10), line=dict(width=4))

    for axis in fig.layout:
        if axis.startswith('xaxis'):
            fig.layout[axis].update(
                        showgrid=False,
                        linecolor='black',
                        linewidth=3,
                        ticks='outside',
                        tickwidth=2,
                        tickcolor='black',
                        ticklen=6,
                        tickformat=".0f"
                    )
        if axis.startswith('yaxis'):
            fig.layout[axis].update(
                showgrid=False,
                linecolor='black',
                linewidth=3,
                ticks='outside',
                tickwidth=2,
                tickcolor='black',
                ticklen=6,
                tickformat=".2f",
                dtick=0.05
            )    

    for annotation in fig.layout.annotations:
        if '=' in annotation.text:
            annotation.text = annotation.text.split('=')[1]
                
    fig.update_layout(
        title=dict(
            x=0,
            xanchor='left'
        ),
        legend=dict(
            orientation='h',
            x=0.48,
            y=1.05,
            xanchor='center',
            yanchor='bottom'
        ),
        margin=dict(t=120)
    )
    return fig

fig = px.line(
    df,
    x="k",
    y="kNN score",
    color="Embedding Space",
    facet_col="metric",
    markers=True,
)
fig = plot_kNN_alignment_scores(fig)

fig.show()

In [27]:
import plotly.express as px
Ks = [3, 5, 10, 50, 100, 200]

collected_results = []
for metric in ['euclidean', 'cosine', 'cityblock']:
    for i_k, k in enumerate(Ks):
        for emb_space in EMB_SPACES:
            results = read_kNN(k, emb_space[0], metric, do_average=False)
            for class_name, (acc, _) in results.items():
                collected_results.append((i_k, emb_space[0], class_name, acc, metric))

df = pd.DataFrame(collected_results, columns=['k', 'Embedding Space', 'Class Name', 'kNN score', 'metric'])

for binding_type in ['BINDING', 'NON-BINDING', 'CRYPTIC-BINDING']:
    binding_df = df[df['Class Name'] == binding_type]
    fig = px.line(
        binding_df,
        x="k",
        y="kNN score",
        color="Embedding Space",
        facet_col="metric",
        markers=True,
    )
    fig.update_layout(
            yaxis_title=f"Mean KNN feature<br>alignment score ({binding_type})",
            template="plotly_white",
            font={"family": "Arial", "color": "black", "size": 12},
            legend_title_text="Embedding models",
            width=1200,
            height=500,
    )

    fig.update_traces(marker=dict(size=10), line=dict(width=4))


    # fig = plot_kNN_alignment_scores(fig)
    for axis in fig.layout:
        if axis.startswith('xaxis'):
            fig.layout[axis].update(
                        showgrid=False,
                        tickmode='array',
                        tickvals=list(range(len(Ks))),
                        ticktext=[str(k) for k in Ks],
                        linecolor='black',
                        linewidth=3,
                        ticks='outside',
                        tickwidth=2,
                        tickcolor='black',
                        ticklen=6,


                    )
        if axis.startswith('yaxis'):
            fig.layout[axis].update(
                showgrid=False,
                tickvals=[binding_df['kNN score'].max(), binding_df['kNN score'].min()],
                tickformat=".2f",
                linecolor='black',
                linewidth=3,
                ticks='outside',
                tickwidth=2,
                tickcolor='black',
                ticklen=6,
            )
    for annotation in fig.layout.annotations:
        if '=' in annotation.text:
            annotation.text = annotation.text.split('=')[1]
                
    fig.update_layout(
        title=dict(
            x=0,
            xanchor='left'
        ),
        legend=dict(
            orientation='h',
            x=0.48,
            y=1.05,
            xanchor='center',
            yanchor='bottom'
        ),
        margin=dict(t=120)
    )

    fig.show()


In [ ]:
import pickle

Ks = [3, 5, 10, 50, 100, 200]

collected_results = []
for metric in ['euclidean', 'cosine', 'cityblock']:
    for k in Ks:
        for emb_space in EMB_SPACES:
            collected_results.append((k, emb_space[0], read_kNN(k, emb_space[0], metric), metric))

df = pd.DataFrame(collected_results, columns=['k', 'Embedding Space', 'kNN score', 'metric'])

performance_results = {}

for emb_space in EMB_SPACES:
    with open(f'{IMG_OUTPUT_PATH}/performance/{emb_space[0]}_torch.pkl', 'rb') as f:
        performance_results[emb_space[0]] = pickle.load(f)

new_collected_results = []
for result in collected_results:
    emb_space = result[1]
    new_collected_results.append(result + (float(performance_results[emb_space]['test_acc']),))
df = pd.DataFrame(new_collected_results, columns=['k', 'Embedding Space', 'kNN score', 'metric', 'ACC'])


In [ ]:
fig = px.line(
    df,
    x="k",
    y="kNN score",
    color="ACC",
    facet_col="metric",
    markers=True,
    color_discrete_sequence=px.colors.sequential.Hot
)
fig = plot_kNN_alignment_scores(fig)

fig.show()

In [50]:
for emb_space in EMB_SPACES:
    with open(f'{IMG_OUTPUT_PATH}/performance/{emb_space[0]}_torch.pkl', 'rb') as f:
        print(pickle.load(f))


{'test_acc': 0.6367182786059604, 'test_roc_auc': 0.7412300986649027, 'test_mcc': 0.17419755486821328, 'test_f1': 0.35036909093307006, 'test_auprc': 0.3843939697400698, 'class_acc_df':              class  accuracy
0          BINDING  0.349225
1  CRYPTIC-BINDING  0.581697
2      NON-BINDING  0.666255}
{'test_acc': 0.6422235378556767, 'test_roc_auc': 0.7395396627546077, 'test_mcc': 0.17394446439436992, 'test_f1': 0.35201078230639266, 'test_auprc': 0.3889438140190236, 'class_acc_df':              class  accuracy
0          BINDING  0.371815
1  CRYPTIC-BINDING  0.588254
2      NON-BINDING  0.670118}
{'test_acc': 0.6290146610088094, 'test_roc_auc': 0.680809189520326, 'test_mcc': 0.13422233116487486, 'test_f1': 0.3352485414566015, 'test_auprc': 0.3756177872231783, 'class_acc_df':              class  accuracy
0          BINDING  0.262641
1  CRYPTIC-BINDING  0.567764
2      NON-BINDING  0.666617}
